# 📊 월별 수입 금액 예측 분석

## 개요
이 노트북은 특정 월 정산 데이터를 기반으로 향후 월별 수입 금액을 예측합니다.

## 분석 흐름
1. **전월 가입자 데이터 로드**: 예) 7월 가입자 (`202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv`)
2. **당월 개통처리부 로드**: 예) 8월 개통처리부 (`2508월 통합 개통처리부 분석.csv`)
3. **요금제 정책 정보 로드**: `MVNO_PRD_PLC.csv`
4. **월별 수입 계산**: 8월, 9월, 10월... 각 월의 예상 수입 금액 산정

## 주요 계산 로직
- **기본 금액**: 정책금 > 기본료 우선순위
- **할인 적용**: 평생할인 + 기간할인(정책기간) + 이벤트가(정책기간)
- **정책 기간**: 정책반영시작일 ~ 정책반영종료일 범위 내에서만 할인 적용
- **월별 누적**: 전월 가입자 + 당월 신규 가입자의 월별 수입 합산

## 입력 파일 형식
- 정산 연월 입력 형식: `YYMM` (예: `2508` = 2025년 8월)
- 전월 가입자: `csv/20{YYMM}_SS001344_ENTR_BY_STACC_PTN_INS_001.csv`
- 당월 개통처리부: `csv/converted/{YYMM}월 통합 개통처리부 분석_전체.csv`
- 요금제 정보: `csv/MVNO_PRD_PLC.csv`


## 1️⃣ 환경 설정 및 라이브러리 로드


In [9]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings
import os
from pathlib import Path
warnings.filterwarnings('ignore')

# 한글 폰트 설정
import matplotlib.font_manager as fm

font_list = [f.name for f in fm.fontManager.ttflist 
             if '한글' in f.name or 'Korean' in f.name or 'Malgun' in f.name 
             or 'Nanum' in f.name or 'Noto' in f.name]

if font_list:
    plt.rcParams['font.family'] = font_list[0]
    print(f"✅ 한글 폰트 설정: {font_list[0]}")
else:
    mac_fonts = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR']
    for font in mac_fonts:
        try:
            plt.rcParams['font.family'] = font
            print(f"✅ 한글 폰트 설정: {font}")
            break
        except:
            continue
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
        print("⚠️ 한글 폰트를 찾을 수 없어 기본 폰트 사용")

plt.rcParams['axes.unicode_minus'] = False

# Pandas 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("\n" + "="*70)
print("📊 월별 수입 금액 예측 프로그램")
print("="*70)
print("✅ 라이브러리 로드 완료!")


✅ 한글 폰트 설정: Noto Sans Carian

📊 월별 수입 금액 예측 프로그램
✅ 라이브러리 로드 완료!


## 2️⃣ 정산 연월 설정 및 파일 경로 구성


In [10]:
# ========================================
# 정산 연월 설정 (YYMM 형식)
# ========================================
SETTLEMENT_YYMM = "2508"  # 2025년 8월 정산

# 파일 경로 자동 구성
year = "20" + SETTLEMENT_YYMM[:2]
month = SETTLEMENT_YYMM[2:]

# 전월 계산
current_date = datetime.strptime(f"{year}{month}", "%Y%m")
prev_month_date = current_date - relativedelta(months=1)
prev_yymm = prev_month_date.strftime("%y%m")
prev_yyyymm = prev_month_date.strftime("%Y%m")

# 파일 경로
BASE_DIR = Path(".")
CSV_DIR = BASE_DIR / "csv"
CONVERTED_DIR = CSV_DIR / "converted"

# 전월 가입자 데이터
prev_month_file = CSV_DIR / f"{prev_yyyymm}_SS001344_ENTR_BY_STACC_PTN_INS_001.csv"

# 당월 개통처리부
current_month_file = CONVERTED_DIR / f"{SETTLEMENT_YYMM}월 통합 개통처리부 분석_전체.csv"

# 요금제 정보
plan_file = CSV_DIR / "MVNO_PRD_PLC.csv"

# 출력 디렉토리
output_dir = BASE_DIR / "output"
output_dir.mkdir(exist_ok=True)

print("\n" + "="*70)
print("📁 파일 경로 설정")
print("="*70)
print(f"정산 연월: {SETTLEMENT_YYMM} ({year}년 {month}월)")
print(f"전월: {prev_yymm} ({prev_month_date.strftime('%Y년 %m월')})")
print(f"\n📂 입력 파일:")
print(f"  - 전월 가입자: {prev_month_file}")
print(f"  - 당월 개통처리부: {current_month_file}")
print(f"  - 요금제 정보: {plan_file}")

# 파일 존재 여부 확인
print(f"\n🔍 파일 존재 여부 확인:")
for file_name, file_path in [
    ("전월 가입자", prev_month_file),
    ("당월 개통처리부", current_month_file),
    ("요금제 정보", plan_file)
]:
    if file_path.exists():
        file_size = file_path.stat().st_size / 1024 / 1024
        print(f"  ✅ {file_name}: 존재 ({file_size:.2f} MB)")
    else:
        print(f"  ❌ {file_name}: 파일이 없습니다!")
        print(f"     경로: {file_path}")



📁 파일 경로 설정
정산 연월: 2508 (2025년 08월)
전월: 2507 (2025년 07월)

📂 입력 파일:
  - 전월 가입자: csv/202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  - 당월 개통처리부: csv/converted/2508월 통합 개통처리부 분석_전체.csv
  - 요금제 정보: csv/MVNO_PRD_PLC.csv

🔍 파일 존재 여부 확인:
  ✅ 전월 가입자: 존재 (264.48 MB)
  ✅ 당월 개통처리부: 존재 (2.03 MB)
  ✅ 요금제 정보: 존재 (0.01 MB)


## 3️⃣ 데이터 로드


In [11]:
def load_csv_with_encoding(file_path, encodings=['utf-8', 'cp949', 'euc-kr', 'utf-8-sig']):
    """다양한 인코딩으로 CSV 파일 로드 시도"""
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
            print(f"  ✅ 로드 성공 (인코딩: {encoding})")
            return df, encoding
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ⚠️  {encoding} 시도 중 오류: {str(e)[:100]}")
            continue
    raise ValueError(f"❌ 파일 로드 실패 - 지원되는 인코딩이 없습니다.")

print("\n" + "="*70)
print("📥 데이터 로드 중...")
print("="*70)

try:
    # 1. 전월 가입자 데이터
    print(f"\n1️⃣ 전월 가입자 데이터 ({prev_yymm})")
    df_prev_month, enc1 = load_csv_with_encoding(prev_month_file)
    print(f"   📊 데이터 크기: {df_prev_month.shape[0]:,}행 × {df_prev_month.shape[1]}열")
    
    # 2. 당월 개통처리부
    print(f"\n2️⃣ 당월 개통처리부 ({SETTLEMENT_YYMM})")
    df_current_month, enc2 = load_csv_with_encoding(current_month_file)
    print(f"   📊 데이터 크기: {df_current_month.shape[0]:,}행 × {df_current_month.shape[1]}열")
    
    # 3. 요금제 정보
    print(f"\n3️⃣ 요금제 정보")
    df_plan, enc3 = load_csv_with_encoding(plan_file)
    print(f"   📊 데이터 크기: {df_plan.shape[0]:,}행 × {df_plan.shape[1]}열")
    
    print("\n" + "="*70)
    print("✅ 모든 데이터 로드 완료!")
    print("="*70)
    
except Exception as e:
    print(f"\n❌ 데이터 로드 중 오류: {e}")
    raise



📥 데이터 로드 중...

1️⃣ 전월 가입자 데이터 (2507)
  ✅ 로드 성공 (인코딩: cp949)
   📊 데이터 크기: 287,638행 × 111열

2️⃣ 당월 개통처리부 (2508)
  ✅ 로드 성공 (인코딩: utf-8)
   📊 데이터 크기: 3,102행 × 107열

3️⃣ 요금제 정보
  ✅ 로드 성공 (인코딩: utf-8)
   📊 데이터 크기: 138행 × 9열

✅ 모든 데이터 로드 완료!


## 4️⃣ 요금제 정보 전처리


In [12]:
print("\n" + "="*70)
print("🔧 요금제 정보 전처리")
print("="*70)

# 요금제 정보 복사
df_plan_processed = df_plan.copy()

# 날짜 컬럼 변환
date_columns = ['정책반영시작일', '정책반영종료일']
for col in date_columns:
    if col in df_plan_processed.columns:
        df_plan_processed[col] = pd.to_datetime(df_plan_processed[col], errors='coerce')
        print(f"✅ {col} 변환 완료")

# 숫자 컬럼 변환
numeric_columns = ['기본료', '평생할인', '기간할인', '이벤트가', '정책금']
for col in numeric_columns:
    if col in df_plan_processed.columns:
        df_plan_processed[col] = pd.to_numeric(df_plan_processed[col], errors='coerce').fillna(0)
        print(f"✅ {col} 변환 완료")

print(f"\n✅ 요금제 정보 전처리 완료: {len(df_plan_processed)}개 요금제")

# 요금제 샘플
print("\n📋 요금제 정보 샘플 (상위 5개):")
print(df_plan_processed.head())



🔧 요금제 정보 전처리
✅ 정책반영시작일 변환 완료
✅ 정책반영종료일 변환 완료
✅ 기본료 변환 완료
✅ 평생할인 변환 완료
✅ 기간할인 변환 완료
✅ 이벤트가 변환 완료
✅ 정책금 변환 완료

✅ 요금제 정보 전처리 완료: 138개 요금제

📋 요금제 정보 샘플 (상위 5개):
        요금제코드                     요금제명  기본료  평생할인  기간할인  이벤트가  정책금    정책반영시작일  \
0  LPZ0002633       [INS]인스 선불정액 300MB    0     0     0     0    0 1900-01-01   
1  LPZ0002635  [INS]인스 선불정액 300MB+(국제)    0     0     0     0    0 1900-01-01   
2  LPZ0002637         [INS]인스 정액선불 10G    0     0     0     0    0 1900-01-01   
3  LPZ0002638       [INS]인스 정액선불 15GB+    0     0     0     0    0 1900-01-01   
4  LPZ0002639        [INS]인스 정액선불 11G+    0     0     0     0    0 1900-01-01   

  정책반영종료일  
0     NaT  
1     NaT  
2     NaT  
3     NaT  
4     NaT  


## 5️⃣ 데이터 병합 (가입자 + 요금제)


In [13]:
print("\n" + "="*70)
print("🔗 데이터 병합")
print("="*70)

# 병합 키 자동 감지
possible_keys = ['MVNO상품코드', '개통요금제코드','상품코드', '요금제코드', 'MVNO_PRD_CD']

# 전월 가입자 병합 키
merge_key_prev = None
for key in possible_keys:
    if key in df_prev_month.columns:
        merge_key_prev = key
        break

# 당월 개통처리부 병합 키
merge_key_current = None
for key in possible_keys:
    if key in df_current_month.columns:
        merge_key_current = key
        break

print(f"\n🔑 병합 키:")
print(f"  - 전월 가입자: {merge_key_prev}")
print(f"  - 당월 개통처리부: {merge_key_current}")

# 전월 가입자 + 요금제 정보 병합
if merge_key_prev:
    print(f"\n1️⃣ 전월 가입자 + 요금제 정보 병합...")
    df_prev_merged = df_prev_month.merge(
        df_plan_processed,
        left_on=merge_key_prev,
        right_on='요금제코드',
        how='left'
    )
    print(f"  ✅ 병합 완료: {len(df_prev_merged):,}행")
    match_rate = (df_prev_merged['요금제명'].notna().sum() / len(df_prev_merged) * 100)
    print(f"  📊 병합 성공률: {match_rate:.1f}%")
else:
    print(f"\n⚠️  전월 가입자 데이터에서 병합 키를 찾을 수 없습니다.")
    df_prev_merged = df_prev_month.copy()

# 당월 개통처리부 + 요금제 정보 병합
if merge_key_current:
    print(f"\n2️⃣ 당월 개통처리부 + 요금제 정보 병합...")
    df_current_merged = df_current_month.merge(
        df_plan_processed,
        left_on=merge_key_current,
        right_on='요금제코드',
        how='left'
    )
    print(f"  ✅ 병합 완료: {len(df_current_merged):,}행")
    match_rate = (df_current_merged['요금제명'].notna().sum() / len(df_current_merged) * 100)
    print(f"  📊 병합 성공률: {match_rate:.1f}%")
else:
    print(f"\n⚠️  당월 개통처리부에서 병합 키를 찾을 수 없습니다.")
    df_current_merged = df_current_month.copy()

print("\n" + "="*70)



🔗 데이터 병합

🔑 병합 키:
  - 전월 가입자: MVNO상품코드
  - 당월 개통처리부: 개통요금제코드

1️⃣ 전월 가입자 + 요금제 정보 병합...
  ✅ 병합 완료: 287,638행
  📊 병합 성공률: 99.9%

2️⃣ 당월 개통처리부 + 요금제 정보 병합...
  ✅ 병합 완료: 3,102행
  📊 병합 성공률: 97.8%



## 6️⃣ 월별 수입 금액 계산 함수


In [14]:
def calculate_monthly_revenue(row, target_month_date):
    """특정 월의 예상 수입 금액 계산"""
    # 기본 금액 결정 (정책금 우선, 없으면 기본료)
    if pd.notna(row.get('정책금', 0)) and row.get('정책금', 0) > 0:
        base_amount = row['정책금']
    else:
        base_amount = row.get('기본료', 0)
    
    # 정책 기간 확인
    policy_start = row.get('정책반영시작일')
    policy_end = row.get('정책반영종료일')
    
    is_policy_period = False
    if pd.notna(policy_start) and pd.notna(policy_end):
        if policy_start <= target_month_date <= policy_end:
            is_policy_period = True
    
    # 할인 계산
    total_discount = 0
    
    # 평생할인 (항상 적용)
    lifetime_discount = row.get('평생할인', 0)
    if pd.notna(lifetime_discount):
        total_discount += lifetime_discount
    
    # 기간할인 (정책 기간에만 적용)
    if is_policy_period:
        period_discount = row.get('기간할인', 0)
        if pd.notna(period_discount):
            total_discount += period_discount
    
    # 이벤트가 (정책 기간에만 적용)
    if is_policy_period:
        event_price = row.get('이벤트가', 0)
        if pd.notna(event_price):
            total_discount += event_price
    
    # 최종 금액 계산 (음수 방지)
    final_amount = max(0, base_amount - total_discount)
    
    return final_amount

def calculate_future_revenue(df, start_month, num_months=12):
    """향후 N개월간의 월별 수입 금액 계산"""
    print(f"\n🔄 월별 수입 금액 계산 시작...")
    print(f"  - 시작 월: {start_month}")
    print(f"  - 계산 기간: {num_months}개월")
    print(f"  - 데이터 크기: {len(df):,}행")
    
    df_result = df.copy()
    start_date = datetime.strptime(start_month, "%Y%m")
    
    # 각 월별로 계산
    for i in range(num_months):
        target_month_date = start_date + relativedelta(months=i)
        month_col = f"M{i+1}_{target_month_date.strftime('%Y%m')}"
        
        print(f"  ⏳ 계산 중: {month_col} ({target_month_date.strftime('%Y년 %m월')})...")
        
        # 각 행별로 수입 계산
        df_result[month_col] = df_result.apply(
            lambda row: calculate_monthly_revenue(row, target_month_date),
            axis=1
        )
    
    print(f"\n✅ 월별 수입 금액 계산 완료!")
    return df_result

print("✅ 월별 수입 금액 계산 함수 정의 완료!")


✅ 월별 수입 금액 계산 함수 정의 완료!


## 7️⃣ 월별 수입 금액 계산 실행


In [15]:
print("\n" + "="*70)
print("💰 월별 수입 금액 계산 실행")
print("="*70)

# 계산 기간 설정
NUM_MONTHS = 12  # 12개월 예측
start_yyyymm = year + month

# 1. 전월 가입자의 당월부터 향후 수입 계산
print(f"\n1️⃣ 전월 가입자의 {SETTLEMENT_YYMM}월부터 {NUM_MONTHS}개월 수입 계산")
df_prev_with_revenue = calculate_future_revenue(df_prev_merged, start_yyyymm, NUM_MONTHS)

# 2. 당월 신규 가입자의 당월부터 향후 수입 계산
print(f"\n2️⃣ 당월 신규 가입자의 {SETTLEMENT_YYMM}월부터 {NUM_MONTHS}개월 수입 계산")
df_current_with_revenue = calculate_future_revenue(df_current_merged, start_yyyymm, NUM_MONTHS)

print("\n" + "="*70)
print("✅ 월별 수입 금액 계산 완료!")
print("="*70)



💰 월별 수입 금액 계산 실행

1️⃣ 전월 가입자의 2508월부터 12개월 수입 계산

🔄 월별 수입 금액 계산 시작...
  - 시작 월: 202508
  - 계산 기간: 12개월
  - 데이터 크기: 287,638행
  ⏳ 계산 중: M1_202508 (2025년 08월)...
  ⏳ 계산 중: M2_202509 (2025년 09월)...
  ⏳ 계산 중: M3_202510 (2025년 10월)...
  ⏳ 계산 중: M4_202511 (2025년 11월)...
  ⏳ 계산 중: M5_202512 (2025년 12월)...
  ⏳ 계산 중: M6_202601 (2026년 01월)...
  ⏳ 계산 중: M7_202602 (2026년 02월)...
  ⏳ 계산 중: M8_202603 (2026년 03월)...
  ⏳ 계산 중: M9_202604 (2026년 04월)...
  ⏳ 계산 중: M10_202605 (2026년 05월)...
  ⏳ 계산 중: M11_202606 (2026년 06월)...
  ⏳ 계산 중: M12_202607 (2026년 07월)...

✅ 월별 수입 금액 계산 완료!

2️⃣ 당월 신규 가입자의 2508월부터 12개월 수입 계산

🔄 월별 수입 금액 계산 시작...
  - 시작 월: 202508
  - 계산 기간: 12개월
  - 데이터 크기: 3,102행
  ⏳ 계산 중: M1_202508 (2025년 08월)...
  ⏳ 계산 중: M2_202509 (2025년 09월)...
  ⏳ 계산 중: M3_202510 (2025년 10월)...
  ⏳ 계산 중: M4_202511 (2025년 11월)...
  ⏳ 계산 중: M5_202512 (2025년 12월)...
  ⏳ 계산 중: M6_202601 (2026년 01월)...
  ⏳ 계산 중: M7_202602 (2026년 02월)...
  ⏳ 계산 중: M8_202603 (2026년 03월)...
  ⏳ 계산 중: M9_202604 (2026년 04월)...
  ⏳ 계산 중: 

## 8️⃣ 월별 총 수입 집계


In [16]:
print("\n" + "="*70)
print("📊 월별 총 수입 집계")
print("="*70)

# 월별 컬럼 추출 (M1_202508 형식만 - 정확한 패턴 매칭)
import re
month_cols = [col for col in df_prev_with_revenue.columns 
              if re.match(r'^M\d+_\d{6}$', col)]

# 월 순서대로 정렬 (M1, M2, ... M12)
month_cols = sorted(month_cols, key=lambda x: int(x.split('_')[0][1:]))

print(f"\n📋 월별 컬럼 확인: {len(month_cols)}개")
if len(month_cols) > 0:
    print(f"  첫 3개: {', '.join(month_cols[:3])}")
    print(f"  마지막 3개: {', '.join(month_cols[-3:])}")

# 전월 가입자의 월별 총 수입
prev_monthly_total = {}
for col in month_cols:
    try:
        total = pd.to_numeric(df_prev_with_revenue[col], errors='coerce').sum()
        prev_monthly_total[col] = total
    except Exception as e:
        print(f"⚠️ {col} 집계 오류: {e}")
        prev_monthly_total[col] = 0

# 당월 신규 가입자의 월별 총 수입
current_monthly_total = {}
for col in month_cols:
    if col in df_current_with_revenue.columns:
        try:
            total = pd.to_numeric(df_current_with_revenue[col], errors='coerce').sum()
            current_monthly_total[col] = total
        except Exception as e:
            print(f"⚠️ {col} 집계 오류: {e}")
            current_monthly_total[col] = 0
    else:
        current_monthly_total[col] = 0

# 전체 총 수입
total_monthly_revenue = {}
for col in month_cols:
    total_monthly_revenue[col] = prev_monthly_total[col] + current_monthly_total[col]

# 결과 출력
print("\n📈 월별 총 수입 예상 금액:")
print("\n" + "-" * 90)
print(f"{'월':<15} {'전월가입자':>18} {'당월신규':>18} {'전체합계':>18} {'증감률':>10}")
print("-" * 90)

prev_total = None
for col in month_cols:
    month_str = col.split('_')[1] if '_' in col else col
    prev_rev = prev_monthly_total[col]
    current_rev = current_monthly_total[col]
    total_rev = total_monthly_revenue[col]
    
    if prev_total is not None and prev_total > 0:
        change_rate = ((total_rev - prev_total) / prev_total) * 100
        change_str = f"{change_rate:+.1f}%"
    else:
        change_str = "-"
    
    print(f"{col:<15} {prev_rev:>15,.0f}원 {current_rev:>15,.0f}원 {total_rev:>15,.0f}원 {change_str:>10}")
    prev_total = total_rev

print("-" * 90)

# 평균 및 총합
avg_prev = sum(prev_monthly_total.values()) / len(prev_monthly_total)
avg_current = sum(current_monthly_total.values()) / len(current_monthly_total)
avg_total = sum(total_monthly_revenue.values()) / len(total_monthly_revenue)

print(f"{'평균 (12개월)':<15} {avg_prev:>15,.0f}원 {avg_current:>15,.0f}원 {avg_total:>15,.0f}원")
print("-" * 90)

sum_prev = sum(prev_monthly_total.values())
sum_current = sum(current_monthly_total.values())
sum_total = sum(total_monthly_revenue.values())

print(f"{'총합 (12개월)':<15} {sum_prev:>15,.0f}원 {sum_current:>15,.0f}원 {sum_total:>15,.0f}원")
print("=" * 90)



📊 월별 총 수입 집계

📋 월별 컬럼 확인: 12개
  첫 3개: M1_202508, M2_202509, M3_202510
  마지막 3개: M10_202605, M11_202606, M12_202607

📈 월별 총 수입 예상 금액:

------------------------------------------------------------------------------------------
월                            전월가입자               당월신규               전체합계        증감률
------------------------------------------------------------------------------------------
M1_202508                     0원               0원               0원          -
M2_202509                     0원               0원               0원          -
M3_202510                     0원               0원               0원          -
M4_202511                     0원               0원               0원          -
M5_202512                     0원               0원               0원          -
M6_202601                     0원               0원               0원          -
M7_202602                     0원               0원               0원          -
M8_202603                     0원               0원    

## 9️⃣ 결과 저장


In [17]:
print("\n" + "="*70)
print("💾 결과 저장")
print("="*70)

# 1. 전월 가입자 상세 데이터 저장
output_file_prev = output_dir / f"revenue_prev_month_{prev_yymm}_detail_{SETTLEMENT_YYMM}.csv"
df_prev_with_revenue.to_csv(output_file_prev, index=False, encoding='utf-8-sig')
print(f"\n1️⃣ 전월 가입자 상세 데이터:")
print(f"  📁 {output_file_prev}")
print(f"  📊 {len(df_prev_with_revenue):,}행 × {len(df_prev_with_revenue.columns)}열")

# 2. 당월 신규 가입자 상세 데이터 저장
output_file_current = output_dir / f"revenue_current_month_{SETTLEMENT_YYMM}_detail.csv"
df_current_with_revenue.to_csv(output_file_current, index=False, encoding='utf-8-sig')
print(f"\n2️⃣ 당월 신규 가입자 상세 데이터:")
print(f"  📁 {output_file_current}")
print(f"  📊 {len(df_current_with_revenue):,}행 × {len(df_current_with_revenue.columns)}열")

# 3. 월별 총 수입 요약 저장
summary_data = []
for col in month_cols:
    month_str = col.split('_')[1] if '_' in col else col
    summary_data.append({
        '월': col,
        '연월': month_str,
        '전월가입자_수입': prev_monthly_total[col],
        '당월신규_수입': current_monthly_total[col],
        '전체_총수입': total_monthly_revenue[col],
        '전월가입자_비율': (prev_monthly_total[col] / total_monthly_revenue[col] * 100) if total_monthly_revenue[col] > 0 else 0,
        '당월신규_비율': (current_monthly_total[col] / total_monthly_revenue[col] * 100) if total_monthly_revenue[col] > 0 else 0
    })

df_summary = pd.DataFrame(summary_data)
output_file_summary = output_dir / f"revenue_summary_{SETTLEMENT_YYMM}.csv"
df_summary.to_csv(output_file_summary, index=False, encoding='utf-8-sig')
print(f"\n3️⃣ 월별 총 수입 요약:")
print(f"  📁 {output_file_summary}")
print(f"  📊 {len(df_summary):,}행")

print("\n" + "="*70)
print("✅ 모든 결과 저장 완료!")
print("="*70)
print(f"\n📂 출력 디렉토리: {output_dir.absolute()}")



💾 결과 저장

1️⃣ 전월 가입자 상세 데이터:
  📁 output/revenue_prev_month_2507_detail_2508.csv
  📊 287,638행 × 132열

2️⃣ 당월 신규 가입자 상세 데이터:
  📁 output/revenue_current_month_2508_detail.csv
  📊 3,102행 × 128열

3️⃣ 월별 총 수입 요약:
  📁 output/revenue_summary_2508.csv
  📊 12행

✅ 모든 결과 저장 완료!

📂 출력 디렉토리: /Users/loveauden/Development/vibe_pandas_project/output


## 🎉 분석 완료

이 노트북은 다음과 같은 작업을 수행했습니다:

1. **정산 연월 설정**: YYMM 형식으로 정산 연월 지정
2. **데이터 로드**: 전월 가입자, 당월 개통처리부, 요금제 정보
3. **데이터 병합**: 가입자 정보와 요금제 정책 매핑
4. **월별 수입 계산**: 12개월간의 예상 수입 금액 산정
5. **결과 저장**: 상세 데이터 및 요약 정보 CSV 파일 생성

### 출력 파일:
- `revenue_prev_month_{YYMM}_detail_{정산YYMM}.csv`: 전월 가입자 상세 수입
- `revenue_current_month_{정산YYMM}_detail.csv`: 당월 신규 가입자 상세 수입
- `revenue_summary_{정산YYMM}.csv`: 월별 총 수입 요약

### 다음 단계:
- 시각화 추가 (matplotlib, seaborn)
- 상품별 수입 분석
- 요금제별 기여도 분석
